# LLM Zoomcamp 2026 — Homework 2: Vector Search

https://courses.datatalks.club/llm-zoomcamp-2026/homework/hw2

We embed the course lesson pages with a lightweight ONNX model
(`Xenova/all-MiniLM-L6-v2`), then search by similarity — by hand, with
`minsearch`, and finally with hybrid search (RRF).

## Answers

| Q | Result | Selected option |
|---|--------|-----------------|
| 1 | `v[0] = -0.0206` | **-0.02** |
| 2 | `cosine = 0.3611` | **0.37** |
| 3 | `02-vector-search/lessons/07-sqlitesearch-vector.md` | same |
| 4 | `04-evaluation/lessons/05-search-metrics.md` | same |
| 5 | `02-vector-search/lessons/08-pgvector.md` | same |
| 6 | `01-agentic-rag/lessons/13-function-calling.md` | same |


## Setup

Run from this folder (`cohorts/2026/02-vector-search/hw2/`), which contains
`download.py` and `embedder.py`.

```bash
uv init --no-workspace
uv add onnxruntime tokenizers numpy tqdm minsearch gitsource
uv add --dev huggingface-hub jupyter
```


In [ ]:
# Download the ONNX model (skips if already present) and load the embedder
import download as dl
dl.download("Xenova/all-MiniLM-L6-v2")

from embedder import Embedder
embedder = Embedder()  # models/Xenova/all-MiniLM-L6-v2
print("Embedder loaded")


## Load the lesson pages (pinned commit `8c1834d`, 72 pages)


In [ ]:
import numpy as np
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import VectorSearch, Index

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]
print(len(documents), "pages")  # 72


## Q1. Embedding a query

Options: `-0.31 / -0.02 / 0.12 / 0.44`


In [ ]:
query = "How does approximate nearest neighbor search work?"
v = embedder.encode(query)
print("Q1  v[0] =", round(float(v[0]), 4), "| shape:", v.shape)


## Q2. Cosine similarity

Options: `0.07 / 0.37 / 0.68 / 0.92`


In [ ]:
page = next(d for d in documents
            if d["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md")
page_vec = embedder.encode(page["content"])
print("Q2  cosine =", round(float(page_vec.dot(v)), 4))  # normalized -> dot == cosine


## Q3. Chunking and search by hand


In [ ]:
chunks = chunk_documents(documents, size=2000, step=1000)
X = np.array(embedder.encode_batch([c["content"] for c in chunks]))
scores = X.dot(v)
print("Q3  file =", chunks[int(scores.argmax())]["filename"])


## Q4. Vector search with minsearch


In [ ]:
vindex = VectorSearch(keyword_fields=[])
vindex.fit(X, chunks)  # fit(vectors_matrix, payload)

q4 = "What metric do we use to evaluate a search engine?"
print("Q4  file =", vindex.search(embedder.encode(q4), num_results=5)[0]["filename"])


## Q5. Text search vs vector search


In [ ]:
tindex = Index(text_fields=["content"], keyword_fields=["filename"])
tindex.fit(chunks)

q5 = "How do I store vectors in PostgreSQL?"
vec5 = [r["filename"] for r in vindex.search(embedder.encode(q5), num_results=5)]
txt5 = [r["filename"] for r in tindex.search(q5, num_results=5)]
print("Q5  in vector not text =", sorted(set(f for f in vec5 if f not in txt5)))


## Q6. Hybrid search (Reciprocal Rank Fusion, k=60)


In [ ]:
def rrf(result_lists, k=60, num_results=5):
    scores, docs = {}, {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc
    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

q6 = "How do I give the model access to tools?"
vr = vindex.search(embedder.encode(q6), num_results=5)
tr = tindex.search(q6, num_results=5)
print("Q6  first after RRF =", rrf([vr, tr])[0]["filename"])
